<h1 align="center">M E D I A &nbsp; S E R V E R</h1>

## Collab Repository Details
- 🔗 **Repo**: https://github.com/kglaynyi/MovieBox
- ⚙️ **Colab Version**: v2 (Heroku-ready)

Run cells top-to-bottom and only fill form fields.

In [ ]:
#@title 1) Install Heroku CLI
!curl -fsSL https://cli-assets.heroku.com/install-ubuntu.sh | sh

In [ ]:
#@title 2) Clone / Open MovieBox
from pathlib import Path
import os

repo_dir = Path('/content/MovieBox')
if not repo_dir.exists():
    !git clone https://github.com/kglaynyi/MovieBox.git /content/MovieBox
else:
    print('Repo already exists at /content/MovieBox')

%cd /content/MovieBox

In [ ]:
#@title 3) Heroku Login
Heroku_Email = "" #@param {type:"string"}
Heroku_API = "" #@param {type:"string"}
Deploy_Git_Name = "Colab Deploy" #@param {type:"string"}

import os

Heroku_Email = Heroku_Email.strip()
Heroku_API = Heroku_API.strip()
Deploy_Git_Name = Deploy_Git_Name.strip() or 'Colab Deploy'

assert Heroku_Email, 'Please enter Heroku email'
assert Heroku_API, 'Please enter Heroku API key'

os.environ['HEROKU_EMAIL'] = Heroku_Email
os.environ['HEROKU_API_KEY'] = Heroku_API

!git config user.email "{Heroku_Email}"
!git config user.name "{Deploy_Git_Name}"
!heroku auth:whoami

In [ ]:
#@title 4) Create Heroku App
App_Name = "" #@param {type:"string"}

App_Name = App_Name.strip()
assert App_Name, 'Please enter a unique Heroku app name'

!heroku apps:info -a {App_Name} || heroku create {App_Name}
!heroku git:remote -a {App_Name}

In [ ]:
#@title 5) Select Python Package Manager (Required by Heroku)
Package_Manager = "pip" #@param ["pip", "uv"]

from pathlib import Path

repo = Path('/content/MovieBox')
requirements = repo / 'requirements.txt'
uv_lock = repo / 'uv.lock'

if Package_Manager == 'pip':
    if uv_lock.exists():
        uv_lock.unlink()
    assert requirements.exists(), 'requirements.txt is missing, cannot deploy with pip'
    print('Using pip (requirements.txt). Removed uv.lock for Heroku compatibility.')
else:
    if requirements.exists():
        requirements.unlink()
    assert uv_lock.exists(), 'uv.lock is missing, cannot deploy with uv'
    print('Using uv (uv.lock). Removed requirements.txt for Heroku compatibility.')

In [ ]:
#@title 6) Prepare Procfile + Runtime
from pathlib import Path

procfile_content = 'web: uvicorn Backend.fastapi.main:app --host 0.0.0.0 --port $PORT\n'
runtime_content = 'python-3.11.9\n'

Path('/content/MovieBox/Procfile').write_text(procfile_content, encoding='utf-8')
Path('/content/MovieBox/runtime.txt').write_text(runtime_content, encoding='utf-8')
print('Created Procfile and runtime.txt')

In [ ]:
#@title 7) Deploy to Heroku
!git add -A
!git commit -m "Prepare Heroku deploy from Colab" || true
!git push heroku HEAD:main -f

In [ ]:
#@title 8) Get Live Link
print(f'✅ Deployed: https://{App_Name}.herokuapp.com')
!heroku open -a {App_Name}